# 2.7 Librerias y modulos

Notebook de practicas para alumnado de 3o de Biotecnologia.

## Objetivo general

Introducir el concepto de modulo, paquete y libreria en Python, y utilizar algunas librerias frecuentes en bioinformatica y analisis de datos: `sys`, `os`, `numpy`, `pandas`, `matplotlib`, `biopython`, `requests`, `scipy` y `scikit-learn`.

## Idea general

Hasta ahora has trabajado sobre todo con estructuras y funciones del propio lenguaje. En esta practica el objetivo es aprender a reutilizar codigo ya escrito por otras personas o por ti mismo. Esa es una de las grandes fortalezas de Python: su ecosistema de modulos y librerias.

## Convenciones de la practica

- Los ejemplos usan nombres biologicos: `genes`, `secuencias`, `expresion`, `condicion`, `longitud`, `calidad`.
- Siempre que sea posible, los datos se generan dentro de la notebook.
- Se crean pequeños ficheros de ejemplo en `data/processed/modulos/`.
- Las soluciones aparecen al final, en una seccion separada.

## Configuracion inicial

Ejecuta esta celda al principio para situar el directorio de trabajo en la raiz del repositorio y crear una carpeta de salida para esta practica.

In [ ]:
from pathlib import Path
import os

candidatos = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
repo_dir = None

for candidato in candidatos:
    if (candidato / "data").exists() and (candidato / "notebooks").exists():
        repo_dir = candidato
        break

if repo_dir is None:
    raise FileNotFoundError("No se ha encontrado la raiz del repositorio")

os.chdir(repo_dir)
salida_dir = Path("data/processed/modulos")
salida_dir.mkdir(parents=True, exist_ok=True)

print("Directorio de trabajo:", Path.cwd())
print("Directorio de salida:", salida_dir.resolve())

## 1. Introduccion: modulos, paquetes y librerias

### Idea basica

Un **modulo** es un fichero `.py` con codigo Python reutilizable.

Un **paquete** es una carpeta que agrupa varios modulos relacionados.

Una **libreria** suele referirse, de forma mas amplia, a un conjunto de modulos y paquetes que resuelven un tipo de problemas.

Ejemplos:

- `math` es un modulo de Python.
- `numpy` es una libreria para calculo numerico.
- `pandas` es una libreria para tablas de datos.
- `Bio` forma parte de Biopython.

### Formas de importar

- `import modulo`
- `import modulo as alias`
- `from modulo import funcion`
- `from modulo import funcion1, funcion2`

Estas formas aparecen constantemente en analisis bioinformatico.

In [ ]:
# Ejemplo 1: distintas formas de importar
import math
import numpy as np
from statistics import mean

print(math.sqrt(16))
print(np.array([1, 2, 3]))
print(mean([2, 4, 6]))

### Ejercicios

1. Explica con tus palabras la diferencia entre modulo y libreria.
2. Escribe tres sentencias `import` distintas: una normal, una con alias y una con `from ... import ...`.

## 2. Crear un modulo propio llamado `biofunciones.py`

Un modulo propio permite guardar funciones reutilizables fuera de la notebook. Eso es una buena practica cuando un bloque de codigo va a utilizarse muchas veces.

En este bloque vamos a crear un modulo con tres funciones:

- `calcular_gc`
- `contar_bases`
- `reverse_complement`

Tambien incluiremos el patron:

- `if __name__ == "__main__":`

Este patron sirve para ejecutar una parte del codigo solo cuando el fichero se lanza como script, pero no cuando se importa como modulo.

In [ ]:
# Ejemplo 2: crear el modulo biofunciones.py
contenido_modulo = '''def calcular_gc(secuencia):
    secuencia = secuencia.upper()
    if len(secuencia) == 0:
        return 0.0
    return 100 * (secuencia.count("G") + secuencia.count("C")) / len(secuencia)

def contar_bases(secuencia):
    secuencia = secuencia.upper()
    return {
        "A": secuencia.count("A"),
        "T": secuencia.count("T"),
        "G": secuencia.count("G"),
        "C": secuencia.count("C")
    }

def reverse_complement(secuencia):
    secuencia = secuencia.upper()
    tabla = str.maketrans("ATGC", "TACG")
    return secuencia.translate(tabla)[::-1]

if __name__ == "__main__":
    ejemplo = "ATGCGCAA"
    print("Secuencia de ejemplo:", ejemplo)
    print("GC:", calcular_gc(ejemplo))
    print("Bases:", contar_bases(ejemplo))
    print("Complemento inverso:", reverse_complement(ejemplo))
'''

ruta_modulo = salida_dir / "biofunciones.py"
ruta_modulo.write_text(contenido_modulo)
print(ruta_modulo)
print(ruta_modulo.read_text()[:300])

In [ ]:
# Ejemplo 3: importar el modulo propio
import sys

if str(salida_dir.resolve()) not in sys.path:
    sys.path.append(str(salida_dir.resolve()))

import biofunciones
from biofunciones import calcular_gc, contar_bases, reverse_complement

secuencia = "ATGCGCAA"
print(biofunciones.calcular_gc(secuencia))
print(contar_bases(secuencia))
print(reverse_complement(secuencia))

### Ejercicios

3. Crea tu propio fichero `biofunciones.py` con las tres funciones indicadas.
4. Importa `calcular_gc` usando `from ... import ...`.
5. Importa el modulo completo y llama a una funcion usando el prefijo del modulo.
6. Explica para que sirve `if __name__ == "__main__":`.

## 3. Pasar argumentos con `sys.argv`

La lista `sys.argv` contiene los argumentos pasados a un script desde la linea de comandos.

Ejemplo general:

- `sys.argv[0]`: nombre del script
- `sys.argv[1]`: primer argumento real
- `sys.argv[2]`: segundo argumento real

Esto es util para convertir un script de Python en una pequeña herramienta reutilizable.

In [ ]:
# Ejemplo 4: crear un script que use sys.argv
contenido_script = '''import sys
from biofunciones import calcular_gc, reverse_complement

def main():
    if len(sys.argv) < 2:
        print("Uso: python analiza_seq.py SECUENCIA")
        return

    secuencia = sys.argv[1].upper()
    print("Secuencia:", secuencia)
    print("Longitud:", len(secuencia))
    print("GC:", calcular_gc(secuencia))
    print("Complemento inverso:", reverse_complement(secuencia))

if __name__ == "__main__":
    main()
'''

ruta_script = salida_dir / "analiza_seq.py"
ruta_script.write_text(contenido_script)
print(ruta_script)

In [ ]:
# Ejemplo 5: ejecutar el script desde Python
import os

os.system(f'python "{ruta_script}" ATGCGTAA')

### Ejercicios

7. Modifica el script para que tambien muestre el conteo de bases.
8. Ejecuta el script con otra secuencia de ejemplo.
9. ¿Que ocurre si no pasas ningun argumento?

## 4. Ejecutar comandos Linux desde Python con `os.system`

`os.system()` permite lanzar comandos de shell desde Python. No es la unica forma de hacerlo, pero es una de las mas simples para empezar.

Es util para automatizar pasos del flujo de trabajo, por ejemplo listar ficheros, crear carpetas, ejecutar scripts o llamar a herramientas externas.

In [ ]:
# Ejemplo 6: listar ficheros con os.system
import os

os.system("ls -lh data/raw | head")

### Ejercicios

10. Usa `os.system()` para crear un directorio de prueba dentro de `data/processed/modulos`.
11. Usa `os.system()` para mostrar las primeras lineas de `data/raw/anotacion_resumen.txt`.
12. Usa `os.system()` para ejecutar tu script `analiza_seq.py` con una secuencia distinta.

## 5. Simular o ejecutar una llamada a `bwa mem`

En bioinformatica es habitual encadenar Python con herramientas externas. Un ejemplo tipico seria el alineamiento con `bwa mem`.

La llamada general es:

```bash
bwa mem referencia.fasta lecturas.fastq > alineamiento.sam
```

En esta practica no vamos a depender de que `bwa` este instalado. Primero comprobaremos si existe y, si no, mostraremos la llamada como simulacion.

In [ ]:
# Ejemplo 7: comprobar si bwa esta disponible
codigo_bwa = os.system("which bwa > /dev/null 2>&1")

comando_bwa = "bwa mem referencia.fasta lecturas.fastq > alineamiento.sam"

if codigo_bwa == 0:
    print("bwa esta instalado. La llamada seria:")
else:
    print("bwa no esta instalado. Se muestra la llamada como simulacion:")

print(comando_bwa)

### Ejercicios

13. Comprueba desde Python si existe `samtools` usando `os.system("which samtools")`.
14. Escribe en una variable una llamada simulada a `bwa mem` con nombres de fichero inventados.
15. Explica por que conviene comprobar antes si un programa externo esta instalado.

## 6. NumPy basico

`NumPy` es una libreria fundamental para trabajo numerico. Su estructura central es el array, que permite trabajar de forma comoda y eficiente con datos numericos.

### Funciones obligatorias de este bloque

- `np.array`
- `np.arange`
- `np.linspace`
- `np.zeros`
- `np.ones`
- `np.mean`, `np.median`, `np.std`, `np.min`, `np.max`, `np.sum`
- `np.random.seed`
- `np.random.normal`
- `np.random.randint`
- `shape`, `ndim`, `dtype`, `reshape`

In [1]:
# Ejemplo 8: arrays sencillos
import numpy as np

longitudes = np.array([900, 1200, 1500, 1800, 2100])
print(longitudes)
print(longitudes.shape)
print(longitudes.ndim)
print(longitudes.dtype)

[ 900 1200 1500 1800 2100]
(5,)
1
int64


In [2]:
# Ejemplo 9: crear arrays de distintas formas
print(np.arange(0, 10, 2))
print(np.linspace(0, 1, 5))
print(np.zeros(4))
print(np.ones(4))

[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]
[0. 0. 0. 0.]
[1. 1. 1. 1.]


In [3]:
# Ejemplo 10: estadisticos basicos
print(np.mean(longitudes))
print(np.median(longitudes))
print(np.std(longitudes))
print(np.min(longitudes))
print(np.max(longitudes))
print(np.sum(longitudes))

1500.0
1500.0
424.26406871192853
900
2100
7500


In [4]:
# Ejemplo 11: simulacion de expresion
np.random.seed(42)
expresion_control = np.random.normal(loc=10, scale=2, size=20)
expresion_tratamiento = np.random.normal(loc=12, scale=2.5, size=20)
conteos = np.random.randint(50, 500, size=10)

print(expresion_control[:5])
print(expresion_tratamiento[:5])
print(conteos)

[10.99342831  9.7234714  11.29537708 13.04605971  9.53169325]
[15.66412192 11.43555925 12.16882051  8.43812953 10.63904319]
[240 451 267  93 211 251 495 319 400 353]


In [ ]:
# Ejemplo 12: reshape y normalizacion simple
matriz = np.arange(12).reshape(3, 4)
print(matriz)

expresion = np.array([5.0, 10.0, 15.0, 20.0])
expresion_normalizada = expresion / np.sum(expresion)
print(expresion_normalizada)

### Ejercicios

16. Crea un array con longitudes de genes y calcula media, mediana, desviacion tipica, minimo y maximo.
17. Crea un array con `np.arange` y otro con `np.linspace`.
18. Genera un array de ceros y otro de unos.
19. Simula dos grupos de expresion, control y tratamiento, con `np.random.normal`.
20. Normaliza un vector de expresion dividiendo cada valor por la suma total.
21. Usa `reshape` para convertir un array lineal en una matriz de 2 filas.

## 7. Pandas basico

`pandas` es la libreria mas usada en Python para trabajar con tablas de datos. Su estructura central es el `DataFrame`, que se parece a una hoja de calculo de Excel o a una tabla SQL.

### Objetivo del bloque

Vamos a crear una tabla sintetica de expresion genica, guardarla como CSV, leerla y explorarla.

In [ ]:
# Ejemplo 13: crear un DataFrame sintetico
import pandas as pd

np.random.seed(123)

genes = [f"gene_{i:02d}" for i in range(1, 21)]
condiciones = ["control"] * 10 + ["tratamiento"] * 10
expresion = np.round(np.random.normal(10, 2, 20), 2)
longitud = np.random.randint(500, 2500, 20)
biotipo = np.random.choice(["enzima", "transportador", "regulador"], size=20)

df_sintetico = pd.DataFrame({
    "gene_id": genes,
    "condition": condiciones,
    "expression": expresion,
    "length": longitud,
    "biotype": biotipo
})

ruta_csv = salida_dir / "expresion_sintetica.csv"
df_sintetico.to_csv(ruta_csv, index=False)
df_sintetico.head()

In [ ]:
# Ejemplo 14: leer y explorar la tabla
df = pd.read_csv(ruta_csv)
print(df.shape)
print(df.columns)
display(df.head())
display(df.tail())
df.info()
display(df.describe(include="all"))

In [ ]:
# Ejemplo 15: seleccion y filtrado
display(df.loc[:, ["gene_id", "expression"]].head())
display(df.iloc[:5, :3])
display(df.query("expression > 10 and length > 1000").head())

In [ ]:
# Ejemplo 16: ordenar, agrupar y asignar columnas
display(df.sort_values("expression", ascending=False).head())
display(df.groupby("condition")["expression"].mean())
display(df["biotype"].value_counts())

df2 = df.assign(expresion_log=df["expression"] / df["expression"].mean())
display(df2.head())

In [ ]:
# Ejemplo 17: exportar una tabla filtrada
df_filtrado = df.query("expression > 10")
ruta_filtrado = salida_dir / "expresion_filtrada.csv"
df_filtrado.to_csv(ruta_filtrado, index=False)
print(ruta_filtrado)

### Ejercicios

22. Crea un DataFrame sintetico parecido al del ejemplo y guardalo como CSV.
23. Leelo con `pd.read_csv` y explora `head`, `tail`, `info`, `describe`, `shape` y `columns`.
24. Filtra genes con `expression > 11`.
25. Calcula la expresion media por `condition` con `groupby`.
26. Cuenta genes por `biotype` con `value_counts`.
27. Añade una nueva columna con `assign`.
28. Exporta a CSV una tabla filtrada.

## 8. Matplotlib basico

`matplotlib` es la libreria basica para crear graficos en Python. Aqui trabajaremos con `matplotlib.pyplot`.

### Funciones obligatorias del bloque

- `plt.figure`
- `plt.plot`
- `plt.scatter`
- `plt.hist`
- `plt.bar`
- `plt.boxplot`
- `plt.xlabel`, `plt.ylabel`, `plt.title`, `plt.legend`, `plt.grid`, `plt.tight_layout`, `plt.savefig`, `plt.show`

In [ ]:
# Ejemplo 18: histograma de longitudes
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.hist(df["length"], bins=8, color="steelblue", edgecolor="black")
plt.xlabel("Longitud")
plt.ylabel("Frecuencia")
plt.title("Histograma de longitudes")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Ejemplo 19: boxplot por condicion
control = df.query("condition == 'control'")["expression"]
tratamiento = df.query("condition == 'tratamiento'")["expression"]

plt.figure(figsize=(5, 4))
plt.boxplot([control, tratamiento], labels=["control", "tratamiento"])
plt.ylabel("Expresion")
plt.title("Expresion por condicion")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Ejemplo 20: scatter longitud vs expresion
plt.figure(figsize=(6, 4))
plt.scatter(df["length"], df["expression"], color="darkgreen", label="genes")
plt.xlabel("Longitud")
plt.ylabel("Expresion")
plt.title("Longitud frente a expresion")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Ejemplo 21: barras y guardado
conteo_biotipo = df["biotype"].value_counts()

plt.figure(figsize=(6, 4))
plt.bar(conteo_biotipo.index, conteo_biotipo.values, color="orange", label="biotipos")
plt.xlabel("Biotipo")
plt.ylabel("Numero de genes")
plt.title("Genes por biotipo")
plt.legend()
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
ruta_figura = salida_dir / "genes_por_biotipo.png"
plt.savefig(ruta_figura, dpi=150)
plt.show()
print(ruta_figura)

In [ ]:
# Ejemplo 22: plot sencillo
plt.figure(figsize=(6, 4))
plt.plot(np.arange(len(df)), df["expression"], marker="o", label="expresion")
plt.xlabel("Indice")
plt.ylabel("Expresion")
plt.title("Expresion por fila")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Ejercicios

29. Crea un histograma de longitudes.
30. Crea un boxplot de expresion por condicion.
31. Crea un scatter de longitud frente a expresion.
32. Crea un grafico de barras con el numero de genes por biotipo.
33. Guarda una de las figuras como PNG con `plt.savefig`.

## 9. Biopython: FASTA y FASTQ

Biopython es una libreria muy utilizada en bioinformatica. Una de sus utilidades mas practicas es leer y escribir formatos de secuencia sin tener que parsearlos manualmente.

### Objetivo del bloque

- crear un FASTA pequeño
- leerlo con `SeqIO.parse` y `SeqIO.read`
- filtrar secuencias y escribir otro FASTA
- crear un FASTQ pequeño con calidades
- leerlo y calcular calidad media

In [ ]:
# Ejemplo 23: crear un FASTA pequeño
fasta_texto = ">seq1 hemoglobina\nATGCGTAAACCC\n>seq2 enzima\nATGCGCGCGCGCGTAA\n>seq3 transportador\nATGAAATTTGGGCCC\n"
ruta_fasta = salida_dir / "secuencias_ejemplo.fasta"
ruta_fasta.write_text(fasta_texto)
print(ruta_fasta.read_text())

In [ ]:
# Ejemplo 24: leer FASTA con Biopython
from Bio import SeqIO

for record in SeqIO.parse(str(ruta_fasta), "fasta"):
    gc = 100 * (record.seq.count("G") + record.seq.count("C")) / len(record.seq)
    print(record.id, record.description, record.seq, len(record.seq), round(gc, 2))

In [ ]:
# Ejemplo 25: leer una sola secuencia con SeqIO.read
ruta_fasta_unico = salida_dir / "secuencia_unica.fasta"
ruta_fasta_unico.write_text(">unica proteina\nATGCGTAA\n")
record = SeqIO.read(str(ruta_fasta_unico), "fasta")
print(record.id)
print(record.description)
print(record.seq)
print(len(record.seq))

In [ ]:
# Ejemplo 26: filtrar secuencias largas y escribir otro FASTA
records_largos = []
for record in SeqIO.parse(str(ruta_fasta), "fasta"):
    if len(record.seq) >= 15:
        records_largos.append(record)

ruta_fasta_largos = salida_dir / "secuencias_largas.fasta"
SeqIO.write(records_largos, str(ruta_fasta_largos), "fasta")
print(ruta_fasta_largos.read_text())

In [ ]:
# Ejemplo 27: crear un FASTQ pequeño con calidades
fastq_texto = "@lectura1\nATGCGTAA\n+\nIIIIHHHH\n@lectura2\nATGCCCAA\n+\nHHHHFFFF\n"
ruta_fastq = salida_dir / "lecturas_ejemplo.fastq"
ruta_fastq.write_text(fastq_texto)
print(ruta_fastq.read_text())

In [ ]:
# Ejemplo 28: leer FASTQ y calcular calidad media
for record in SeqIO.parse(str(ruta_fastq), "fastq"):
    calidad = record.letter_annotations["phred_quality"]
    calidad_media = sum(calidad) / len(calidad)
    print(record.id, record.seq, calidad, round(calidad_media, 2))

### Ejercicios

34. Crea un FASTA pequeño dentro de la notebook.
35. Leelo con `SeqIO.parse` y calcula longitud y GC por secuencia.
36. Filtra secuencias largas y guardalas en otro FASTA con `SeqIO.write`.
37. Crea un FASTQ pequeño con calidades.
38. Leelo y calcula la calidad media por lectura usando `record.letter_annotations["phred_quality"]`.

## 10. Requests: descargar secuencias de UniProt

`requests` es una libreria muy comoda para hacer peticiones HTTP. En bioinformatica es util para interactuar con APIs y descargar recursos remotos.

### Objetivo del bloque

Vamos a intentar descargar una secuencia FASTA de UniProt usando el identificador `P69905`. Si no hay internet o si la respuesta no es correcta, la notebook no debe fallar.

In [ ]:
# Ejemplo 29: peticion simple con requests
import requests

url_fasta = "https://rest.uniprot.org/uniprotkb/P69905.fasta"
response = requests.get(url_fasta, timeout=20)

print(response.status_code)
print(response.ok)
print(response.text[:120])

In [ ]:
# Ejemplo 30: descarga robusta y lectura posterior
ruta_uniprot = salida_dir / "proteina_uniprot.fasta"

try:
    response = requests.get(url_fasta, timeout=20)
    if response.status_code == 200 and response.ok:
        ruta_uniprot.write_text(response.text)
        print("Descarga correcta")
    else:
        print("No se pudo descargar la secuencia. Codigo:", response.status_code)
except Exception as e:
    print("Error de conexion o descarga:", e)

if ruta_uniprot.exists():
    record_uni = SeqIO.read(str(ruta_uniprot), "fasta")
    aminoacidos = str(record_uni.seq)
    frecuencias = {aa: aminoacidos.count(aa) for aa in sorted(set(aminoacidos))}
    print(record_uni.id)
    print(len(record_uni.seq))
    print(frecuencias)

In [ ]:
# Ejemplo 31: uso basico de response.json con una API publica sencilla
url_json = "https://rest.uniprot.org/uniprotkb/search?query=P69905&format=json&size=1"

try:
    response_json = requests.get(url_json, timeout=20)
    print(response_json.status_code)
    if response_json.ok:
        datos_json = response_json.json()
        print(type(datos_json))
        print(list(datos_json.keys())[:5])
except Exception as e:
    print("No se pudo recuperar JSON:", e)

### Ejercicios

39. Intenta descargar la secuencia FASTA de `P69905` con `requests.get`.
40. Comprueba `response.status_code`, `response.ok` y los primeros caracteres de `response.text`.
41. Guarda la secuencia como `proteina_uniprot.fasta`.
42. Leela con Biopython y calcula su longitud y la frecuencia de aminoacidos.
43. Añade gestion de errores para que no falle la practica si no hay internet.

## 11. SciPy: tests estadisticos basicos

`SciPy` incluye muchas herramientas cientificas. Aqui usaremos el submodulo `stats` para pruebas estadisticas basicas.

### Interpretacion basica del p-valor

Un p-valor pequeño sugiere que los datos observados son poco compatibles con la hipotesis nula. En un contexto introductorio, suele interpretarse que si `p < 0.05` hay evidencia de diferencia o asociacion estadisticamente significativa.

In [ ]:
# Ejemplo 32: pruebas basicas con SciPy
from scipy import stats

print(stats.shapiro(expresion_control))
print(stats.ttest_ind(expresion_control, expresion_tratamiento))
print(stats.mannwhitneyu(expresion_control, expresion_tratamiento))

In [ ]:
# Ejemplo 33: correlaciones
longitudes_small = df["length"].to_numpy()
expresion_small = df["expression"].to_numpy()

print(stats.pearsonr(longitudes_small, expresion_small))
print(stats.spearmanr(longitudes_small, expresion_small))

### Ejercicios

44. Comprueba la normalidad de `expresion_control` y `expresion_tratamiento` con `stats.shapiro`.
45. Compara ambos grupos con `stats.ttest_ind`.
46. Repite la comparacion con `stats.mannwhitneyu`.
47. Calcula la correlacion entre longitud y expresion con `stats.pearsonr` y `stats.spearmanr`.
48. Escribe una interpretacion breve de los p-valores obtenidos.

## 12. Scikit-learn: clasificacion simple

`scikit-learn` es la libreria de aprendizaje automatico mas usada en Python para tareas clasicas. En este bloque haremos una clasificacion binaria muy sencilla.

### Objetivo

Crear un conjunto de datos sintetico con dos variables:

- `expression`
- `length`

y una etiqueta binaria:

- `control`
- `tratamiento`

In [ ]:
# Ejemplo 34: construir un dataset sintetico
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

np.random.seed(7)
n = 80

expression_ml = np.concatenate([
    np.random.normal(9, 1.5, n // 2),
    np.random.normal(12, 1.5, n // 2)
])

length_ml = np.concatenate([
    np.random.normal(1200, 200, n // 2),
    np.random.normal(1700, 250, n // 2)
])

y = np.array([0] * (n // 2) + [1] * (n // 2))
X = np.column_stack([expression_ml, length_ml])

print(X[:5])
print(y[:5])

In [ ]:
# Ejemplo 35: train/test, escalado, entrenamiento y evaluacion
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

modelo = LogisticRegression()
modelo.fit(X_train_scaled, y_train)

y_pred = modelo.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred)
matriz = confusion_matrix(y_test, y_pred)

print("Accuracy:", acc)
print("Confusion matrix:\n", matriz)

### Interpretacion basica

- `accuracy_score` indica la proporcion de predicciones correctas.
- `confusion_matrix` muestra aciertos y errores separados por clase.

En este nivel, lo importante no es optimizar el modelo, sino entender el flujo completo: datos, division en entrenamiento y prueba, escalado, ajuste, prediccion y evaluacion.

### Ejercicios

49. Crea un dataset sintetico con `expression` y `length`.
50. Divide en train/test con `train_test_split`.
51. Escala con `StandardScaler`.
52. Entrena un modelo `LogisticRegression`.
53. Predice sobre test y calcula `accuracy_score` y `confusion_matrix`.
54. Escribe una interpretacion breve del resultado.

## 13. Mini-reto final integrador

Este reto busca combinar ideas de toda la notebook. No hace falta usar todas las librerias a la vez, pero si integrar varias de ellas en un flujo razonable.

### Tareas del mini-reto

1. Descargar o leer una secuencia.
2. Calcular metricas basicas.
3. Crear una tabla `pandas`.
4. Hacer una figura.
5. Aplicar un test estadistico simple.
6. Guardar resultados.

### Enunciado del mini-reto

55. Diseña un pequeño flujo de trabajo que haga lo siguiente:

- lea una o varias secuencias desde un FASTA local o descargado
- calcule longitud y GC
- almacene los resultados en un `DataFrame`
- dibuje al menos una figura con `matplotlib`
- compare dos grupos con un test sencillo de `scipy.stats`
- guarde la tabla y la figura en `data/processed/modulos/`

El objetivo no es que el codigo sea largo, sino que sea claro, correcto y bien organizado.

## Soluciones

Las soluciones siguientes son orientativas. En programacion suele haber varias formas correctas de resolver el mismo problema.

### Solucion orientativa: bloque de modulos

In [ ]:
# Solucion orientativa de importacion y modulo propio
from biofunciones import calcular_gc, contar_bases, reverse_complement
print(calcular_gc("ATGCGTAA"))
print(contar_bases("ATGCGTAA"))
print(reverse_complement("ATGCGTAA"))

### Solucion orientativa: NumPy

In [ ]:
longitudes_sol = np.array([800, 950, 1100, 1300, 1600])
print(np.mean(longitudes_sol), np.median(longitudes_sol), np.std(longitudes_sol))
print(np.min(longitudes_sol), np.max(longitudes_sol), np.sum(longitudes_sol))

### Solucion orientativa: Pandas

In [ ]:
display(df.query("expression > 11").head())
display(df.groupby("condition")["expression"].mean())
display(df["biotype"].value_counts())

### Solucion orientativa: Biopython

In [ ]:
for record in SeqIO.parse(str(ruta_fasta), "fasta"):
    gc = 100 * (record.seq.count("G") + record.seq.count("C")) / len(record.seq)
    print(record.id, len(record.seq), round(gc, 2))

### Solucion orientativa: SciPy

In [ ]:
resultado_t = stats.ttest_ind(expresion_control, expresion_tratamiento)
resultado_u = stats.mannwhitneyu(expresion_control, expresion_tratamiento)
print(resultado_t)
print(resultado_u)

### Solucion orientativa: scikit-learn

In [ ]:
print("Accuracy:", acc)
print("Matriz de confusion:\n", matriz)

## Cierre

En esta practica has dado el salto desde Python como lenguaje a Python como ecosistema. Has creado un modulo propio, ejecutado scripts con argumentos, interactuado con el sistema operativo, trabajado con arrays, tablas, figuras, secuencias biologicas, peticiones web, estadistica y clasificacion.

Ese salto es muy importante en bioinformatica: a partir de aqui, aprender una libreria nueva suele consistir en identificar que tipo de problema resuelve y como integrarla en un flujo de trabajo reproducible.